# Diffusion SILK 시각화

`diffusion_silk_notebook.ipynb`에서 학습한 diffusion 모델의 예측 결과를 GT와 나란히
3D 애니메이션으로 비교합니다. 실제 MANO(등록된 모델)로 6D 회전 -> 관절 위치를 복원합니다.

## 1. 환경 설정

In [1]:

import os, math, pickle, random, inspect, io
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from google.colab import drive
drive.mount('/content/drive')

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


Mounted at /content/drive
device: cuda


## 2. chumpy/MANO 설정

In [2]:

!pip install chumpy -q

if not hasattr(np, "bool"): np.bool = bool
if not hasattr(np, "int"): np.int = int
if not hasattr(np, "float"): np.float = float
if not hasattr(np, "object"): np.object = object
if not hasattr(np, "str"): np.str = str
if not hasattr(np, "complex"): np.complex = complex
if not hasattr(np, "unicode"): np.unicode = str
if not hasattr(inspect, "getargspec"): inspect.getargspec = inspect.getfullargspec

import chumpy
print("chumpy 로드 성공")

!pip install -q smplx
import smplx

ROOT = "/content/drive/MyDrive/KUBIG/KUBIG SUMMER CONTEST"
MANO_MODEL_ROOT = ROOT  # mano 폴더의 부모 경로

print("mano 폴더 존재:", os.path.exists(f"{MANO_MODEL_ROOT}/mano"))
if os.path.exists(f"{MANO_MODEL_ROOT}/mano"):
    print("내용물:", os.listdir(f"{MANO_MODEL_ROOT}/mano"))

mano_right = smplx.create(
    model_path=MANO_MODEL_ROOT, model_type="mano", is_rhand=True,
    use_pca=False, flat_hand_mean=False, num_pca_comps=45,
).to(DEVICE)

print("MANO 로드 완료")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 1.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


/tmp/ipykernel_554/1284326284.py:6: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"): np.object = object
/tmp/ipykernel_554/1284326284.py:7: FutureWarning: In the future `np.str` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "str"): np.str = str


chumpy 로드 성공
mano 폴더 존재: True
내용물: ['MANO_RIGHT.pkl', 'MANO_LEFT.pkl']
MANO 로드 완료


## 3. 설정값 (학습 노트북과 동일해야 체크포인트 로드 가능)

In [3]:

N_JOINTS = 15
POSE_DIM = N_JOINTS * 6  # 90

C_CONTEXT = 10
EVAL_T_VALUES = [5, 10, 20, 30]
MAX_SEQ_LEN = C_CONTEXT + 30 + 1  # 41

D_MODEL = 1024
N_HEADS = 8
N_LAYERS = 6
D_FF = 4096
DROPOUT = 0.1

N_DIFFUSION_STEPS = 1000


## 4. 6D/flip 유틸리티 (학습 노트북과 동일, row 기준)

In [4]:

_LEFT_HAND_FLIP_MASK = np.array([
    [ 1.0, -1.0, -1.0],
    [-1.0,  1.0,  1.0],
    [-1.0,  1.0,  1.0],
], dtype=np.float32)


def rotation_6d_to_matrix_np(d6):
    a1, a2 = d6[..., :3], d6[..., 3:]
    b1 = a1 / np.linalg.norm(a1, axis=-1, keepdims=True)
    b2 = a2 - np.sum(b1 * a2, axis=-1, keepdims=True) * b1
    b2 = b2 / np.linalg.norm(b2, axis=-1, keepdims=True)
    b3 = np.cross(b1, b2, axis=-1)
    return np.stack((b1, b2, b3), axis=-2)


def matrix_to_rotation_6d_np(mats):
    batch_dim = mats.shape[:-2]
    return mats[..., :2, :].copy().reshape(*batch_dim, 6)


def flip_left_hand_features(left_feats, n_joints=N_JOINTS):
    left = left_feats.astype(np.float32, copy=False)
    T_len = left.shape[0]
    pose_6d = left[:, :n_joints * 6].reshape(T_len * n_joints, 6)
    mats = rotation_6d_to_matrix_np(pose_6d) * _LEFT_HAND_FLIP_MASK
    flipped = matrix_to_rotation_6d_np(mats).reshape(T_len, n_joints * 6)
    if left.shape[-1] > n_joints * 6:
        return np.concatenate([flipped, left[:, n_joints * 6:]], axis=-1)
    return flipped


def rotation_6d_to_matrix(d6):
    a1, a2 = d6[..., 0:3], d6[..., 3:6]
    b1 = torch.nn.functional.normalize(a1, dim=-1)
    b2 = a2 - (b1 * a2).sum(-1, keepdim=True) * b1
    b2 = torch.nn.functional.normalize(b2, dim=-1)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-2)


def matrix_to_axis_angle(R):
    batch_shape = R.shape[:-2]
    R_flat = R.reshape(-1, 3, 3)
    cos_theta = ((R_flat[:, 0, 0] + R_flat[:, 1, 1] + R_flat[:, 2, 2]) - 1) / 2
    cos_theta = cos_theta.clamp(-1 + 1e-7, 1 - 1e-7)
    theta = torch.acos(cos_theta)
    axis = torch.stack([
        R_flat[:, 2, 1] - R_flat[:, 1, 2],
        R_flat[:, 0, 2] - R_flat[:, 2, 0],
        R_flat[:, 1, 0] - R_flat[:, 0, 1],
    ], dim=-1)
    denom = (2 * torch.sin(theta)).clamp(min=1e-7).unsqueeze(-1)
    axis = axis / denom
    return (axis * theta.unsqueeze(-1)).reshape(*batch_shape, 3)


def sixd_sequence_to_axis_angle(seq_6d):
    if isinstance(seq_6d, np.ndarray):
        seq_6d = torch.from_numpy(seq_6d).float()
    return matrix_to_axis_angle(rotation_6d_to_matrix(seq_6d))


@torch.no_grad()
def mano_forward(mano_layer, hand_pose_aa, device=DEVICE):
    '''MANO 내부 파라미터가 기본 requires_grad=True라 no_grad로 감싸야 함
    (안 그러면 .cpu().numpy() 시점에 RuntimeError).'''
    T = hand_pose_aa.shape[0]
    global_orient = torch.zeros(T, 3, device=device)
    hand_pose = hand_pose_aa.reshape(T, -1).to(device)
    betas = torch.zeros(T, 10, device=device)
    output = mano_layer(global_orient=global_orient, hand_pose=hand_pose, betas=betas, return_verts=True)
    return output.joints  # (T, 16, 3)


MANO_PARENT = [-1, 0,1,2, 0,4,5, 0,7,8, 0,10,11, 0,13,14]
MANO_BONE_JOINTS = [i for i in range(16) if MANO_PARENT[i] != -1]


## 5. 모델 클래스 (학습 노트북과 동일 — 체크포인트 로드용)

In [5]:

class RelativePositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200):
        super().__init__()
        self.max_len = max_len
        self.pos_embedding = nn.Embedding(2 * max_len + 1, d_model)

    def forward(self, x, rel_pos):
        idx = (rel_pos + self.max_len).clamp(0, 2 * self.max_len)
        return x + self.pos_embedding(idx)


class TimestepEmbedding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model), nn.SiLU(), nn.Linear(d_model, d_model),
        )

    def forward(self, t):
        half = self.d_model // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t.float()[:, None] * freqs[None]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return self.mlp(emb)


class DiffusionSILKHand(nn.Module):
    def __init__(self, pose_dim=POSE_DIM, d_model=D_MODEL, n_heads=N_HEADS,
                 n_layers=N_LAYERS, d_ff=D_FF, dropout=DROPOUT, max_len=MAX_SEQ_LEN):
        super().__init__()
        self.input_proj = nn.Linear(pose_dim + 1, d_model)
        self.pos_enc = RelativePositionalEncoding(d_model, max_len=max_len)
        self.time_emb = TimestepEmbedding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, pose_dim)

    def forward(self, x_noisy_with_flag, rel_pos, t, valid_mask=None):
        h = self.input_proj(x_noisy_with_flag)
        h = self.pos_enc(h, rel_pos)
        h = h + self.time_emb(t).unsqueeze(1)
        key_padding_mask = ~valid_mask if valid_mask is not None else None
        h = self.encoder(h, src_key_padding_mask=key_padding_mask)
        return self.output_proj(h)


## 6. 샘플링(추론) 함수 — Flow Matching, ODE Euler 적분

DDPM의 1000스텝(50스텝 DDIM 서브샘플링)을 Flow Matching으로 교체 — 노이즈에서 데이터로
가는 **직선 경로의 속도장(velocity)**을 예측해서 Euler 적분으로 샘플링합니다. 모션
도메인에서 diffusion보다 훨씬 적은 스텝(4~8스텝 근처에서 포화)으로 비슷하거나 더 나은
품질을 낸다는 문헌 근거로 채택 — 학습 노트북에서 실제로 이전 최선(키프레임 고도화,
DDPM) 대비 전 지표 개선 확인됨. `schedule` 객체 자체가 필요 없어짐.


In [6]:

@torch.no_grad()
def sample_flow(model, target, obs_mask, rel_pos, valid_mask,
                 device=DEVICE, n_steps=20, use_amp=True):
    '''ODE Euler 적분. 관측 구간(컨텍스트/목표/보너스)은 매 스텝 정답으로 강제 치환
    (CondMDI의 inpainting 메커니즘 그대로 유지).'''
    B, L, _ = target.shape
    x = torch.randn(B, L, POSE_DIM, device=device)
    dt = 1.0 / n_steps

    obs_mask_f = obs_mask.unsqueeze(-1).float()

    for i in range(n_steps):
        t_val = i * dt
        t = torch.full((B,), t_val, device=device)

        x_input = x * (1 - obs_mask_f) + target * obs_mask_f
        flag = obs_mask.float().unsqueeze(-1)
        model_in = torch.cat([x_input, flag], dim=-1)

        if use_amp and device.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                v_pred = model(model_in, rel_pos, t, valid_mask)
        else:
            v_pred = model(model_in, rel_pos, t, valid_mask)
        v_pred = v_pred.float()

        x = x + v_pred * dt   # Euler step

    x = x * (1 - obs_mask_f) + target * obs_mask_f
    return x


## 7. 데이터 로더 (팀원1 인덱스 + LMDB + flip + segment)

In [7]:

!pip install -q lmdb
import lmdb

INDEX_DIR = f"{ROOT}/SignSparK_index"
DATA_ROOT = "/content/signspark_local_data"


def open_lmdb(split):
    path = f"{DATA_ROOT}/lmdb/{split}/How2Sign_reopt_{split}.lmdb"
    env = lmdb.open(path, readonly=True, lock=False, readahead=False, meminit=False, max_readers=1024)
    return env


def load_clip_raw(env, clip_id):
    with env.begin() as txn:
        raw = txn.get(clip_id.encode() if isinstance(clip_id, str) else clip_id)
    npz = np.load(io.BytesIO(raw), allow_pickle=True)
    return {
        "segment": npz["segment"],
        "left_features": npz["left_features"][:, :POSE_DIM],
        "right_features": npz["right_features"][:, :POSE_DIM],
    }


class DiffusionSignSparkDataset(Dataset):
    def __init__(self, split, index_dir=INDEX_DIR, mode="eval", seed=SEED):
        self.split = split
        self.mode = mode
        self.rng = np.random.default_rng(seed)

        idx_path = f"{index_dir}/{split}_index.npz"
        data = np.load(idx_path, allow_pickle=True)
        self.clip_ids = data["clip_ids"]
        self.clip_idx = data["clip_idx"]
        self.hand = data["hand"]
        self.start = data["start"]
        self.T_arr = data["T"] if "T" in data.files else None

        self.env = None
        self._clip_cache = {}

    def _get_env(self):
        if self.env is None:
            self.env = open_lmdb(self.split)
        return self.env

    def _get_clip(self, clip_i):
        if clip_i not in self._clip_cache:
            if len(self._clip_cache) > 500:
                self._clip_cache.clear()
            cid = self.clip_ids[clip_i]
            self._clip_cache[clip_i] = load_clip_raw(self._get_env(), cid)
        return self._clip_cache[clip_i]

    def __del__(self):
        if self.env is not None:
            try:
                self.env.close()
            except Exception:
                pass

    def __len__(self):
        return len(self.clip_idx)

    def get_window(self, i, T=None):
        '''시각화용: (feats_window, segment_window, hand_flag) 반환. T 지정 안 하면 인덱스의 T 사용.'''
        clip_i = int(self.clip_idx[i])
        hand_flag = int(self.hand[i])
        start = int(self.start[i])
        clip = self._get_clip(clip_i)

        key = "left_features" if hand_flag == 1 else "right_features"
        feats = clip[key]
        if hand_flag == 1:
            feats = flip_left_hand_features(feats)

        if T is None:
            T = int(self.T_arr[i]) if self.T_arr is not None else 15

        L = C_CONTEXT + T + 1
        window = feats[start:start + L].astype(np.float32)
        segment_window = clip["segment"][start:start + L]
        return window, segment_window, hand_flag, T


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 14.5 MB/s eta 0:00:00


In [8]:
import os, shutil

DRIVE_BACKUP = f"{ROOT}/SignSparK_lmdb_backup"

def setup_lmdb():
    os.makedirs(DATA_ROOT, exist_ok=True)
    drive_lmdb_dir = f"{DRIVE_BACKUP}/lmdb"
    local_lmdb_dir = f"{DATA_ROOT}/lmdb"

    def _all_present(base_dir):
        return all(
            os.path.exists(f"{base_dir}/{split}/How2Sign_reopt_{split}.lmdb/data.mdb")
            for split in ["train", "dev", "test"]
        )

    if _all_present(local_lmdb_dir):
        print("로컬에 이미 LMDB 있음, 그대로 사용")
        return

    if _all_present(drive_lmdb_dir):
        print("Drive 백업에서 로컬로 복사 중...")
        shutil.copytree(drive_lmdb_dir, local_lmdb_dir, dirs_exist_ok=True)
        print("복사 완료")
        return

    print("Drive 백업도 없음 -- 처음부터 다운로드")
    os.environ["HF_HUB_DISABLE_XET"] = "1"
    if not os.path.exists("/content/SignSparK_repo"):
        os.system("git clone -q https://github.com/JianHe0628/SignSparK.git /content/SignSparK_repo")
    os.system(f'cd /content/SignSparK_repo && python tools/download_data.py '
              f'--datasets How2Sign --dest "{DATA_ROOT}"')

    print("Drive에 백업 중...")
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    shutil.copytree(local_lmdb_dir, drive_lmdb_dir, dirs_exist_ok=True)
    print("백업 완료")


setup_lmdb()

for split in ["train", "dev", "test"]:
    path = f"{DATA_ROOT}/lmdb/{split}/How2Sign_reopt_{split}.lmdb/data.mdb"
    exists = os.path.exists(path)
    print(f"{split}: {exists}")

Drive 백업에서 로컬로 복사 중...
복사 완료
train: True
dev: True
test: True


## 8. 다이나믹한(=격렬하게 움직이는) 예시 자동 탐색

랜덤으로 고르면 정적인 구간만 걸릴 수 있으니, 팀원 1의 방식(평균 회전 속도 기준)을
참고해서 **일정 임계값 이상 움직이는 클립을 자동으로 찾습니다.**


In [9]:

def rotation_geodesic_distance(R1, R2):
    R_rel = torch.matmul(R1.transpose(-1, -2), R2)
    trace = R_rel[..., 0, 0] + R_rel[..., 1, 1] + R_rel[..., 2, 2]
    cos_theta = ((trace - 1) / 2).clamp(-1 + 1e-7, 1 - 1e-7)
    return torch.acos(cos_theta)


def find_dynamic_window(ds, T=20, n_try=300, min_avg_speed_deg=8.0, seed=0):
    '''평균 회전 속도가 min_avg_speed_deg(도/프레임) 이상인 gap 구간을 가진 인덱스를 찾음.
    없으면 지금까지 본 것 중 가장 다이나믹한 인덱스를 반환.'''
    rng = np.random.default_rng(seed)
    if ds.T_arr is not None:
        candidates = np.where(ds.T_arr == T)[0]
    else:
        candidates = np.arange(len(ds))
    tried = rng.choice(candidates, size=min(n_try, len(candidates)), replace=False)

    best_idx, best_speed = None, -1.0
    for i in tried:
        window, seg, hand_flag, T_actual = ds.get_window(int(i), T=T)
        seq_t = torch.from_numpy(window.reshape(-1, N_JOINTS, 6)).float()
        R_all = rotation_6d_to_matrix(seq_t)
        gap_R = R_all[C_CONTEXT:C_CONTEXT + T_actual]
        if gap_R.shape[0] < 2:
            continue
        angles = rotation_geodesic_distance(gap_R[:-1], gap_R[1:])
        avg_speed = torch.rad2deg(angles).mean().item()
        if avg_speed > best_speed:
            best_speed, best_idx = avg_speed, int(i)
        if avg_speed >= min_avg_speed_deg:
            print(f"찾음: idx={i}, 평균 속도={avg_speed:.2f}도/프레임")
            return int(i)

    print(f"임계값({min_avg_speed_deg}도) 이상은 못 찾음. 가장 다이나믹한 것 사용: "
          f"idx={best_idx}, 속도={best_speed:.2f}도/프레임")
    return best_idx


## 9. GT/예측을 MANO 위치로 복원 + 애니메이션

In [10]:

def get_gt_pred_window_mano(model, ds, idx, T=20, n_steps=20, mano_layer=None,
                             use_oracle_bonus=False):
    '''use_oracle_bonus: 기본 False -- 진짜 배포 상황(gap 안 정답을 전혀 모름)을 정직하게
    보여주려면 순수 컨텍스트+목표만 써야 함. segment==2 보너스는 학습 때만 쓰는 것이고,
    여기서 True로 켜면 "정보가 더 있을 때의 상한선"을 참고용으로 보는 것 -- 절대 메인
    결과와 섞으면 안 됨.'''
    window, seg, hand_flag, T_actual = ds.get_window(idx, T=T)
    L = window.shape[0]

    target = torch.from_numpy(window).unsqueeze(0).float().to(DEVICE)  # (1, L, 90)
    obs_mask = torch.zeros(1, L, dtype=torch.bool).to(DEVICE)
    obs_mask[0, :C_CONTEXT] = True
    obs_mask[0, -1] = True

    if use_oracle_bonus:
        gap_range = slice(C_CONTEXT, L - 1)
        bonus_idx = np.where(seg[gap_range] == 2)[0] + C_CONTEXT
        obs_mask[0, bonus_idx] = True

    valid_mask = torch.ones(1, L, dtype=torch.bool).to(DEVICE)
    rel_pos = (torch.arange(L) - (L - 1)).unsqueeze(0).to(DEVICE)

    model.eval()
    pred = sample_flow(model, target, obs_mask, rel_pos, valid_mask, n_steps=n_steps)

    gt_aa = sixd_sequence_to_axis_angle(target[0].reshape(L, N_JOINTS, 6))
    pred_aa = sixd_sequence_to_axis_angle(pred[0].reshape(L, N_JOINTS, 6))

    gt_joints = mano_forward(mano_layer, gt_aa).cpu().numpy()
    pred_joints = mano_forward(mano_layer, pred_aa).cpu().numpy()

    gap_mask_np = (~obs_mask[0]).cpu().numpy() & valid_mask[0].cpu().numpy()
    return gt_joints, pred_joints, gap_mask_np


def animate_hand_comparison(gt_joints, pred_joints, gap_mask, interval=120):
    fig = plt.figure(figsize=(10, 5))
    ax_gt = fig.add_subplot(121, projection='3d')
    ax_pred = fig.add_subplot(122, projection='3d')

    all_pts = np.concatenate([gt_joints, pred_joints], axis=0)
    lims = [(all_pts[..., i].min(), all_pts[..., i].max()) for i in range(3)]

    def draw(ax, points, color, title):
        ax.clear()
        ax.set_xlim(lims[0]); ax.set_ylim(lims[1]); ax.set_zlim(lims[2])
        ax.set_title(title)
        ax.scatter(points[:, 0], points[:, 1], points[:, 2], c=color, s=25)
        for j in MANO_BONE_JOINTS:
            p = MANO_PARENT[j]
            ax.plot([points[j, 0], points[p, 0]], [points[j, 1], points[p, 1]],
                     [points[j, 2], points[p, 2]], c=color)

    def update(t):
        is_gap = gap_mask[t]
        draw(ax_gt, gt_joints[t], 'tab:blue', f"GT (frame {t}{' [GAP]' if is_gap else ''})")
        draw(ax_pred, pred_joints[t], 'tab:red' if is_gap else 'tab:blue',
             f"Diffusion 예측 (frame {t}{' [GAP]' if is_gap else ''})")

    anim = FuncAnimation(fig, update, frames=gt_joints.shape[0], interval=interval)
    plt.close(fig)
    return HTML(anim.to_jshtml())


## 10. 체크포인트 로드

⚠️ **`ckpt_path`가 Flow Matching으로 재학습한 체크포인트를 가리키는지 확인하세요.**
DDPM 버전과 파일명이 같으면 덮어썼을 수 있고, 다르게 저장했다면 경로를 맞춰야 합니다
(예: `diffusion_silk_best.pt` vs `diffusion_silk_flow_best.pt`). DDPM 가중치를 Flow
Matching 샘플러(`sample_flow`)에 잘못 넣으면 에러 없이 조용히 의미 없는 결과가 나옵니다.


In [11]:

model = DiffusionSILKHand().to(DEVICE)
ckpt_path = f"{ROOT}/diffusion_silk_best.pt"
print("체크포인트 존재:", os.path.exists(ckpt_path))

ckpt = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"불러온 체크포인트 epoch: {ckpt.get('epoch')}, best_val: {ckpt.get('best_val')}")


/tmp/ipykernel_554/3834622.py:40: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


체크포인트 존재: True
불러온 체크포인트 epoch: 29, best_val: 0.40567924269132544


## 11. 실행 — 다이나믹한 예시 찾아서 시각화

In [12]:

test_ds = DiffusionSignSparkDataset("test", mode="eval")

dynamic_idx = find_dynamic_window(test_ds, T=20, n_try=300, min_avg_speed_deg=8.0)

# 순수 성능(정직한 버전) -- use_oracle_bonus 기본값 False
gt_joints, pred_joints, gap_mask = get_gt_pred_window_mano(
    model, test_ds, dynamic_idx, T=20, n_steps=20, mano_layer=mano_right
)
animate_hand_comparison(gt_joints, pred_joints, gap_mask)


Output hidden; open in https://colab.research.google.com to view.

### (참고, 선택) 오라클 버전과 비교

`use_oracle_bonus=True`로 같은 인덱스를 다시 그리면, "gap 안의 세그먼트 시작점을
안다면 얼마나 더 좋아지는지"를 정성적으로 참고할 수 있습니다. **메인 결과로는 절대
쓰면 안 되고, 순수 버전과의 차이를 눈으로 확인하는 용도로만 사용합니다.**


In [13]:

gt_joints_oracle, pred_joints_oracle, gap_mask_oracle = get_gt_pred_window_mano(
    model, test_ds, dynamic_idx, T=20, n_steps=20, mano_layer=mano_right, use_oracle_bonus=True
)
animate_hand_comparison(gt_joints_oracle, pred_joints_oracle, gap_mask_oracle)


/tmp/ipykernel_554/3316751207.py:62: UserWarning: Glyph 50696 (\N{HANGUL SYLLABLE YE}) missing from font(s) DejaVu Sans.
  return HTML(anim.to_jshtml())
/tmp/ipykernel_554/3316751207.py:62: UserWarning: Glyph 52769 (\N{HANGUL SYLLABLE CEUG}) missing from font(s) DejaVu Sans.
  return HTML(anim.to_jshtml())
